<a href="https://colab.research.google.com/github/bashar730/heart-disease-ai/blob/streamlit-docs/notebooks/04_Streamlit_Interface_SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# تحميل النموذج النهائي مباشرة من GitHub

from pathlib import Path
import requests

MODEL_URL = (
    "https://raw.githubusercontent.com/"
    "bashar730/heart-disease-ai/"
    "model-evaluation/models/heart_disease_model.pkl"
)

MODEL_PATH = Path("/content/heart_disease_model.pkl")

response = requests.get(MODEL_URL, timeout=60)

if response.status_code == 200 and len(response.content) > 1000:
    MODEL_PATH.write_bytes(response.content)

    print("✅ تم تحميل النموذج من GitHub بنجاح.")
    print("مسار النموذج:", MODEL_PATH)
    print("حجم الملف:", MODEL_PATH.stat().st_size, "بايت")
else:
    print("❌ تعذّر تحميل النموذج.")
    print("رمز الاستجابة:", response.status_code)
    print("حجم المحتوى:", len(response.content))

✅ تم تحميل النموذج من GitHub بنجاح.
مسار النموذج: /content/heart_disease_model.pkl
حجم الملف: 2245 بايت


In [ ]:
# تثبيت المكتبات اللازمة لبناء الواجهة واستخدام SHAP

!pip install -q streamlit shap==0.52.0 scikit-learn==1.6.1 joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 79.6 MB/s eta 0:00:00


In [ ]:
# تحميل النموذج وفحص محتوياته

import joblib

bundle = joblib.load(MODEL_PATH)

model = bundle["model"]
FEATURES = list(bundle["feature_names"])

print("✅ تم فتح ملف النموذج بنجاح.")
print("\nنوع النموذج:", type(model).__name__)
print("خطوات النموذج:", list(model.named_steps.keys()))
print("عدد الخصائص:", len(FEATURES))

print("\nأسماء الخصائص وترتيبها:")
for number, feature in enumerate(FEATURES, start=1):
    print(number, "-", feature)

print("\nإعدادات Logistic Regression:")
print(model.named_steps["logistic"])

✅ تم فتح ملف النموذج بنجاح.

نوع النموذج: Pipeline
خطوات النموذج: ['scaler', 'logistic']
عدد الخصائص: 13

أسماء الخصائص وترتيبها:
1 - age
2 - sex
3 - cp
4 - trestbps
5 - chol
6 - fbs
7 - restecg
8 - thalach
9 - exang
10 - oldpeak
11 - slope
12 - ca
13 - thal

إعدادات Logistic Regression:
LogisticRegression(C=10, class_weight='balanced', max_iter=2000,
                   random_state=42, solver='liblinear')


In [ ]:
# إنشاء مفسر SHAP واختباره على حالة واحدة

import pandas as pd
import numpy as np
import shap

scaler = model.named_steps["scaler"]
logistic_model = model.named_steps["logistic"]

# استخدام متوسطات التدريب المحفوظة داخل StandardScaler كمرجع
background_raw = pd.DataFrame(
    [scaler.mean_],
    columns=FEATURES
)

background_scaled = scaler.transform(background_raw)

masker = shap.maskers.Independent(
    background_scaled,
    max_samples=1
)

shap_explainer = shap.LinearExplainer(
    logistic_model,
    masker
)

# حالة تجريبية بنفس ترتيب الخصائص
test_case = pd.DataFrame([{
    "age": 63,
    "sex": 1,
    "cp": 1,
    "trestbps": 145,
    "chol": 233,
    "fbs": 1,
    "restecg": 2,
    "thalach": 150,
    "exang": 0,
    "oldpeak": 2.3,
    "slope": 3,
    "ca": 0,
    "thal": 6
}])

prediction = int(model.predict(test_case)[0])
probability = float(model.predict_proba(test_case)[0, 1])

scaled_case = scaler.transform(test_case)
explanation = shap_explainer(scaled_case)
shap_values = explanation.values[0]

# التأكد أن مجموع SHAP يساوي مخرج النموذج الخطي
shap_total = explanation.base_values[0] + shap_values.sum()
model_output = logistic_model.decision_function(scaled_case)[0]

print("✅ تم إنشاء مفسر SHAP بنجاح.")
print("قرار النموذج:", prediction)
print(f"احتمال الفئة الإيجابية: {probability * 100:.2f}%")
print("اختبار تطابق SHAP مع النموذج:", np.isclose(shap_total, model_output))

print("\nتأثير الخصائص حسب SHAP:")
ranked = sorted(
    zip(FEATURES, shap_values),
    key=lambda item: abs(item[1]),
    reverse=True
)

for feature, value in ranked:
    direction = "يرفع التقدير" if value > 0 else "يخفض التقدير"
    print(f"{feature:10s} | {value: .4f} | {direction}")

✅ تم إنشاء مفسر SHAP بنجاح.
قرار النموذج: 0
احتمال الفئة الإيجابية: 33.04%
اختبار تطابق SHAP مع النموذج: True

تأثير الخصائص حسب SHAP:
cp         | -1.2982 | يخفض التقدير
slope      |  0.8366 | يرفع التقدير
ca         | -0.8247 | يخفض التقدير
fbs        | -0.6744 | يخفض التقدير
sex        |  0.4486 | يرفع التقدير
thal       |  0.4100 | يرفع التقدير
exang      | -0.3344 | يخفض التقدير
trestbps   |  0.3132 | يرفع التقدير
oldpeak    |  0.2946 | يرفع التقدير
restecg    |  0.2340 | يرفع التقدير
age        | -0.1112 | يخفض التقدير
chol       | -0.0715 | يخفض التقدير
thalach    | -0.0079 | يخفض التقدير


In [ ]:
# إنشاء هيكل مشروع الواجهة ونسخ النموذج إليه

from pathlib import Path
import shutil

PROJECT_PATH = Path("/content/BNN_Heart_Disease_App")
APP_FOLDER = PROJECT_PATH / "app"
MODELS_FOLDER = PROJECT_PATH / "models"

APP_FOLDER.mkdir(parents=True, exist_ok=True)
MODELS_FOLDER.mkdir(parents=True, exist_ok=True)

FINAL_MODEL_PATH = MODELS_FOLDER / "heart_disease_model.pkl"

shutil.copy2(
    MODEL_PATH,
    FINAL_MODEL_PATH
)

print("✅ تم إنشاء هيكل التطبيق بنجاح.")
print("\nمجلد المشروع:", PROJECT_PATH)
print("مجلد الواجهة:", APP_FOLDER)
print("ملف النموذج:", FINAL_MODEL_PATH)
print("النموذج موجود:", FINAL_MODEL_PATH.exists())

✅ تم إنشاء هيكل التطبيق بنجاح.

مجلد المشروع: /content/BNN_Heart_Disease_App
مجلد الواجهة: /content/BNN_Heart_Disease_App/app
ملف النموذج: /content/BNN_Heart_Disease_App/models/heart_disease_model.pkl
النموذج موجود: True


In [ ]:
%%writefile /content/BNN_Heart_Disease_App/app/app.py

from pathlib import Path
import joblib
import pandas as pd
import shap
import streamlit as st

# إعداد صفحة التطبيق
st.set_page_config(
    page_title="نظام BNN لأمراض القلب",
    page_icon="🫀",
    layout="centered",
    initial_sidebar_state="collapsed"
)

# تنسيق الواجهة ودعم اللغة العربية
st.markdown("""
<style>
[data-testid="stSidebar"], [data-testid="stSidebarCollapsedControl"] {
    display: none;
}
.block-container {
    max-width: 720px;
    padding: 1rem 1rem 3rem;
}
html, body, [class*="st-"] {
    direction: rtl;
    text-align: right;
}
.title-card {
    background: linear-gradient(135deg, #14213d, #27496d);
    color: white;
    padding: 24px;
    border-radius: 20px;
    text-align: center;
    margin-bottom: 18px;
}
.title-card h1 {
    font-size: 1.7rem;
    margin: 0 0 8px;
}
.title-card p {
    margin: 0;
    color: #dce8f5;
}
.result {
    padding: 22px;
    border-radius: 18px;
    text-align: center;
    margin-top: 18px;
}
.low {
    background: #ecfdf3;
    border: 1px solid #61c985;
    color: #17663a;
}
.high {
    background: #fff1f2;
    border: 1px solid #ef7b88;
    color: #9f2434;
}
.percent {
    font-size: 2.6rem;
    font-weight: 800;
    direction: ltr;
}
.note {
    background: #fff8e8;
    border: 1px solid #efd27c;
    padding: 14px;
    border-radius: 14px;
    margin-top: 18px;
    color: #72520c;
}
</style>
""", unsafe_allow_html=True)

# أسماء الخصائص الطبية وترجمتها
LABELS = {
    "age": "العمر",
    "sex": "الجنس",
    "cp": "نوع ألم الصدر",
    "trestbps": "ضغط الدم أثناء الراحة",
    "chol": "الكوليسترول",
    "fbs": "سكر الصيام",
    "restecg": "تخطيط القلب",
    "thalach": "أقصى معدل لضربات القلب",
    "exang": "ذبحة المجهود",
    "oldpeak": "انخفاض مقطع ST",
    "slope": "ميل مقطع ST",
    "ca": "عدد الأوعية الرئيسية",
    "thal": "اختبار الثاليوم"
}

FEATURES = list(LABELS)
MODEL_PATH = (
    Path(__file__).resolve().parents[1]
    / "models"
    / "heart_disease_model.pkl"
)

# تحميل النموذج وإنشاء مفسر SHAP مرة واحدة
@st.cache_resource(show_spinner=False)
def load_resources():
    bundle = joblib.load(MODEL_PATH)

    if list(bundle.get("feature_names", [])) != FEATURES:
        raise ValueError("خصائص النموذج لا تطابق خصائص الواجهة.")

    model = bundle["model"]
    scaler = model.named_steps["scaler"]
    logistic = model.named_steps["logistic"]

    background_raw = pd.DataFrame(
        [scaler.mean_],
        columns=FEATURES
    )
    background_scaled = scaler.transform(background_raw)

    masker = shap.maskers.Independent(
        background_scaled,
        max_samples=1
    )
    explainer = shap.LinearExplainer(logistic, masker)

    return model, explainer

# عرض الاختيارات العربية وإرجاع الرمز الرقمي للنموذج
def choose(label, options, default, key):
    codes = list(options)
    return st.selectbox(
        label,
        codes,
        index=codes.index(default),
        format_func=options.get,
        key=key
    )

# حساب أهم العوامل المؤثرة باستخدام SHAP
def explain_result(model, explainer, data):
    scaled_data = model.named_steps["scaler"].transform(data)
    values = explainer(scaled_data).values[0]

    ranked = sorted(
        zip(FEATURES, values),
        key=lambda item: abs(item[1]),
        reverse=True
    )

    increase = [(name, value) for name, value in ranked if value > 0][:3]
    decrease = [(name, value) for name, value in ranked if value < 0][:3]
    return increase, decrease

# إعادة الحقول إلى القيم الافتراضية
def clear_form():
    for key in FEATURES:
        st.session_state.pop(key, None)

SEX = {1: "ذكر", 0: "أنثى"}
CP = {
    1: "ذبحة نموذجية",
    2: "ذبحة غير نموذجية",
    3: "ألم غير ذبحي",
    4: "بدون أعراض"
}
YES_NO = {0: "لا", 1: "نعم"}
ECG = {0: "طبيعي", 1: "شذوذ ST-T", 2: "تضخم البطين الأيسر"}
SLOPE = {1: "صاعد", 2: "مسطح", 3: "هابط"}
CA = {0: "صفر", 1: "وعاء واحد", 2: "وعاءان", 3: "ثلاثة أوعية"}
THAL = {3: "طبيعي", 6: "عيب ثابت", 7: "عيب قابل للعكس"}

# عنوان التطبيق
st.markdown("""
<div class="title-card">
<h1>🫀 نظام BNN للتنبؤ بأمراض القلب</h1>
<p>أدخل المؤشرات الطبية للحصول على تقدير تعليمي وتفسير باستخدام SHAP</p>
</div>
""", unsafe_allow_html=True)

# نموذج إدخال الخصائص الثلاث عشرة
with st.form("heart_form"):
    st.subheader("المعلومات الأساسية")
    age = st.number_input("العمر (سنة)", 18, 100, 50, key="age")
    sex = choose("الجنس", SEX, 1, "sex")
    cp = choose("نوع ألم الصدر", CP, 1, "cp")

    st.subheader("القياسات والفحوصات")
    trestbps = st.number_input(
        "ضغط الدم أثناء الراحة (mmHg)", 70, 250, 120, key="trestbps"
    )
    chol = st.number_input(
        "الكوليسترول (mg/dL)", 80, 700, 200, key="chol"
    )
    fbs = choose("هل سكر الصيام أكبر من 120؟", YES_NO, 0, "fbs")
    restecg = choose("نتيجة تخطيط القلب", ECG, 0, "restecg")

    st.subheader("اختبار الجهد والقلب")
    thalach = st.number_input(
        "أقصى معدل لضربات القلب", 40, 230, 150, key="thalach"
    )
    exang = choose("هل ظهرت ذبحة بسبب المجهود؟", YES_NO, 0, "exang")
    oldpeak = st.number_input(
        "انخفاض مقطع ST", 0.0, 10.0, 1.0, 0.1, key="oldpeak"
    )
    slope = choose("ميل مقطع ST", SLOPE, 1, "slope")
    ca = choose("عدد الأوعية الرئيسية", CA, 0, "ca")
    thal = choose("نتيجة اختبار الثاليوم", THAL, 3, "thal")

    first_button, second_button = st.columns(2)

    with first_button:
        submitted = st.form_submit_button(
            "تحليل البيانات",
            type="primary",
            use_container_width=True
        )

    with second_button:
        st.form_submit_button(
            "مسح الحقول",
            on_click=clear_form,
            use_container_width=True
        )

# تنفيذ التنبؤ والتفسير
if submitted:
    try:
        values = [
            age, sex, cp, trestbps, chol, fbs, restecg,
            thalach, exang, oldpeak, slope, ca, thal
        ]

        data = pd.DataFrame([values], columns=FEATURES)
        model, explainer = load_resources()

        probability = float(model.predict_proba(data)[0, 1])
        prediction = int(model.predict(data)[0])

        css_class = "high" if prediction == 1 else "low"
        status = (
            "احتمال مرتفع وفق النموذج"
            if prediction == 1
            else "احتمال منخفض وفق النموذج"
        )
        icon = "⚠️" if prediction == 1 else "✅"

        st.markdown(f"""
        <div class="result {css_class}">
        <div class="percent">{probability * 100:.1f}%</div>
        <h2>{icon} {status}</h2>
        <p>هذه نتيجة تقديرية تعليمية وليست تشخيصًا طبيًا.</p>
        </div>
        """, unsafe_allow_html=True)

        st.progress(probability)

        increase, decrease = explain_result(
            model, explainer, data
        )

        increase_text = "، ".join(
            LABELS[name] for name, value in increase
        )
        decrease_text = "، ".join(
            LABELS[name] for name, value in decrease
        )

        st.subheader("تفسير النتيجة باستخدام SHAP")
        st.success("عوامل رفعت تقدير النموذج: " + increase_text)
        st.info("عوامل خفّضت تقدير النموذج: " + decrease_text)

        st.caption(
            "قِيَم SHAP توضّح تأثير الخصائص داخل النموذج، "
            "ولا تمثل أسبابًا طبية أو علاقة سببية."
        )

    except Exception as error:
        st.error(f"تعذّر إجراء التنبؤ: {error}")

# التنبيه الطبي الثابت
st.markdown("""
<div class="note">
<b>تنبيه طبي:</b>
هذا النظام مشروع تعليمي لدعم القرار، ولا يستبدل الطبيب
أو الفحوصات الطبية المعتمدة.
</div>
""", unsafe_allow_html=True)

Overwriting /content/BNN_Heart_Disease_App/app/app.py


In [ ]:
# فحص سلامة كود الواجهة

from pathlib import Path
import py_compile

APP_FILE = Path(
    "/content/BNN_Heart_Disease_App/app/app.py"
)

try:
    py_compile.compile(
        str(APP_FILE),
        doraise=True
    )

    code_lines = len(
        APP_FILE.read_text(encoding="utf-8").splitlines()
    )

    print("✅ كود الواجهة سليم ولا توجد أخطاء Syntax.")
    print("مسار ملف الواجهة:", APP_FILE)
    print("عدد أسطر الكود:", code_lines)
    print("ملف النموذج موجود:", FINAL_MODEL_PATH.exists())

except Exception as error:
    print("❌ يوجد خطأ داخل كود الواجهة:")
    print(error)

✅ كود الواجهة سليم ولا توجد أخطاء Syntax.
مسار ملف الواجهة: /content/BNN_Heart_Disease_App/app/app.py
عدد أسطر الكود: 287
ملف النموذج موجود: True


In [ ]:
%%writefile /content/BNN_Heart_Disease_App/requirements.txt

streamlit
pandas
numpy>=2,<3
joblib
scikit-learn==1.6.1
shap==0.52.0

Writing /content/BNN_Heart_Disease_App/requirements.txt


In [ ]:
# تشغيل تطبيق Streamlit داخل Google Colab

import subprocess
import time
from pathlib import Path
from google.colab.output import eval_js

APP_FILE = Path(
    "/content/BNN_Heart_Disease_App/app/app.py"
)

LOG_FILE = Path("/content/streamlit_log.txt")

log_output = open(
    LOG_FILE,
    "w",
    encoding="utf-8"
)

streamlit_process = subprocess.Popen(
    [
        "streamlit",
        "run",
        str(APP_FILE),
        "--server.port",
        "8501",
        "--server.headless",
        "true"
    ],
    stdout=log_output,
    stderr=subprocess.STDOUT
)

print("جاري تشغيل التطبيق...")
time.sleep(10)

if streamlit_process.poll() is None:
    APP_URL = eval_js(
        "google.colab.kernel.proxyPort(8501)"
    )

    print("✅ تم تشغيل التطبيق بنجاح.")
    print("\nافتح رابط التطبيق:")
    print(APP_URL)

else:
    log_output.close()
    print("❌ لم يعمل التطبيق. هذا سجل الخطأ:\n")
    print(LOG_FILE.read_text(encoding="utf-8"))

جاري تشغيل التطبيق...
✅ تم تشغيل التطبيق بنجاح.

افتح رابط التطبيق:
https://8501-m-s-kkb-ass1c2-3q8k70faf0jgr-c.asia-southeast1-2.prod.colab.dev


In [ ]:
# فتح تطبيق Streamlit عبر رابط Cloudflare مؤقت

import os
import re
import time
import subprocess
import urllib.request
from pathlib import Path
from IPython.display import display, HTML

APP_PATH = "/content/BNN_Heart_Disease_App/app/app.py"
PORT = 8501

STREAMLIT_LOG = Path("/content/streamlit_app.log")
TUNNEL_LOG = Path("/content/cloudflared.log")
CLOUDFLARED = Path("/content/cloudflared")


def app_is_ready():
    try:
        with urllib.request.urlopen(
            f"http://127.0.0.1:{PORT}/_stcore/health",
            timeout=3
        ) as response:
            return response.status == 200
    except Exception:
        return False


# تشغيل Streamlit إذا لم يكن يعمل
if not app_is_ready():

    subprocess.run(
        ["fuser", "-k", f"{PORT}/tcp"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    log_file = open(STREAMLIT_LOG, "w")

    streamlit_process = subprocess.Popen(
        [
            "streamlit", "run", APP_PATH,
            f"--server.port={PORT}",
            "--server.address=0.0.0.0",
            "--server.headless=true",
            "--server.enableCORS=false",
            "--server.enableXsrfProtection=false",
            "--browser.gatherUsageStats=false"
        ],
        stdout=log_file,
        stderr=subprocess.STDOUT
    )

    log_file.close()

    for _ in range(30):
        if app_is_ready():
            break
        time.sleep(1)


if not app_is_ready():
    print("❌ لم يعمل Streamlit.")
    if STREAMLIT_LOG.exists():
        print(STREAMLIT_LOG.read_text(errors="ignore")[-4000:])
    raise RuntimeError("تعذّر تشغيل التطبيق")


print("✅ تطبيق Streamlit يعمل محليًا.")


# تنزيل أداة Cloudflare Tunnel
if not CLOUDFLARED.exists() or CLOUDFLARED.stat().st_size < 1_000_000:

    subprocess.run(
        [
            "wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", str(CLOUDFLARED)
        ],
        check=True
    )

    os.chmod(CLOUDFLARED, 0o755)


# إيقاف أي نفق قديم
subprocess.run(
    ["pkill", "-x", "cloudflared"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# تشغيل نفق جديد
tunnel_file = open(TUNNEL_LOG, "w")

cloudflare_process = subprocess.Popen(
    [
        str(CLOUDFLARED),
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}"
    ],
    stdout=tunnel_file,
    stderr=subprocess.STDOUT
)

tunnel_file.close()


# انتظار ظهور الرابط
public_url = None

for _ in range(45):
    time.sleep(1)

    if TUNNEL_LOG.exists():
        log_text = TUNNEL_LOG.read_text(errors="ignore")

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            log_text
        )

        if match:
            public_url = match.group(0)
            break


if public_url:

    print("\n✅ تم إنشاء رابط التطبيق:")
    print(public_url)

    display(
        HTML(
            f"""
            <a href="{public_url}" target="_blank"
               style="
                   display:inline-block;
                   padding:14px 25px;
                   background:#0f766e;
                   color:white;
                   text-decoration:none;
                   border-radius:10px;
                   font-size:18px;
                   font-weight:bold;
                   margin-top:10px;">
                افتح تطبيق أمراض القلب
            </a>
            """
        )
    )

else:
    print("❌ لم يظهر رابط Cloudflare.")
    print(TUNNEL_LOG.read_text(errors="ignore")[-4000:])

✅ تطبيق Streamlit يعمل محليًا.

✅ تم إنشاء رابط التطبيق:
https://blessed-ranges-batman-biotechnology.trycloudflare.com


In [ ]:

import shutil
from pathlib import Path
from google.colab import files

project_path = Path("/content/BNN_Heart_Disease_App")

zip_path = shutil.make_archive(
    "/content/BNN_Heart_Disease_App_SHAP",
    "zip",
    root_dir="/content",
    base_dir="BNN_Heart_Disease_App"
)

print("✅ تم تجهيز نسخة التطبيق:")
print(zip_path)

files.download(zip_path)

✅ تم تجهيز نسخة التطبيق:
/content/BNN_Heart_Disease_App_SHAP.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>